# Ouvertures et victoires successives — projet original complété

**Question principale : après une victoire, gagner la suivante est-il plus fréquent si l’on répète l’ouverture ?**

Ce notebook conserve les sections I–V du rapport d’origine comme base et comparaison Hot Hand. Les anciens blocs remplacés sont conservés en commentaires `##`. Les commentaires historiques contenant des résultats sont repliés et explicitement signalés : ils ne sont pas validés pour la nouvelle collecte.

La **partie VI** répond au nouvel objectif. Elle distingue les parties immédiatement successives observées et la prochaine partie de même couleur, qui peut être plus éloignée. Les nulles restent dans la chronologie et interrompent une suite de victoires. Une partie courte/inconnue n’est pas forcée dans une ouverture.

**Par défaut, DEMO=True utilise des données simulées pour rendre le notebook exécutable sans identifiants. Pour tes résultats, choisir DEMO=False et renseigner LOCAL_FILE ou S3_RUN_ID dans I.2.**

Les fichiers privés .env et les tokens ne doivent jamais être ajoutés au dépôt. S3 utilise le bucket Hadrien défini dans config_projet.py.

Limites conservées : collecte historique potentiellement incomplète, dépendance par joueur, adversaires partagés et absence d’interprétation causale. Les sessions historiques utilisent les dates de début, pas une vraie pause entre fin et début.

<details><summary>Introduction du projet initial Hot Hand</summary>

# Hot Hand dans les parties d’échecs en ligne

Ce notebook étudie l’existence d’un effet de *Hot Hand* dans des parties d’échecs en ligne, entendu comme :

> **une victoire augmente-t-elle la probabilité de gagner la partie suivante ?**

L’analyse s’appuie sur des séquences de parties jouées par les mêmes joueurs. Elle progresse en quatre étapes :
1. **Dépendances brutes** entre résultats consécutifs (global, par format, par niveau).
2. **Proximité temporelle** : distinction entre simples parties “consécutives” et vraies séquences **intra-session**.
3. **Mémoire** : test de dépendance à court terme via une approche markovienne (ordre 1 vs ordre 2) sur les sessions.
4. **Contrôles et mécanismes** : régressions (niveau, couleur), fatigue, répétition d’ouverture, et test par permutation (null model).

> 💡 **Note de lecture**  
> Le notebook est conçu pour être lisible comme un rapport : chaque résultat important est commenté juste après le tableau ou le graphique correspondant.

## Sommaire

- **I. Récupération, préparation et prise en main des données**
  - I.1 Fonctions utilitaires
  - I.2 Chargement et nettoyage
  - I.3 Statistiques descriptives
- **II. Dépendances brutes**
  - II.1 Résultats consécutifs (test global)
  - II.2 Par format de partie
  - II.3 Par niveau (tranches de rating)
- **III. Proximité temporelle et sessions de jeu**
  - III.1 Définition d’une session
  - III.2 Comparaison : brut vs intra-session
  - III.3 Lecture et implications
- **IV. Mémoire intra-session : analyse markovienne**
  - IV.1 Probabilités conditionnelles (ordre 2)
  - IV.2 Test formel : ordre 1 vs ordre 2
  - IV.3 Interprétation
- **V. Contrôles et mécanismes**
  - V.1 Régression logistique (niveau, couleur)
  - V.2 Fatigue et dynamique de session
  - V.3 Ouverture rejouée
  - V.4 Null model (permutation)
  - V.5 Conclusion de section
- **Conclusion générale**

</details>


## Installation

In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## import numpy as np
## import pandas as pd
## import matplotlib.pyplot as plt
## from scipy.stats import chi2_contingency
## from scipy.stats import chi2
## import statsmodels.formula.api as smf
## import statsmodels.api as sm
## import os
## import s3fs
## FIN ANCIEN BLOC

## Sur Colab : récupérer le code du dépôt, puis installer ses dépendances.
## Sur ton ordinateur : le notebook se lance depuis la racine du projet.
from pathlib import Path
import os
import sys
import subprocess

if not Path("config_projet.py").exists():
    if "google.colab" not in sys.modules:
        raise FileNotFoundError("Ouvrir le notebook depuis la racine du projet")
    target = Path("/content/projet_ouvertures")
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/Hadrienfonrojet/ANALYSE-MARCHE-.git", str(target)], check=True)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, chi2
import statsmodels.formula.api as smf
import statsmodels.api as sm
import config_projet as cfg
from IPython.display import display, Image


## I. Récupération, préparation et prise en main des données 

### I.1 Fonctions utilitaires

In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## # === CHARGEMENT DES DONNÉES ===
## def load_games_from_s3(bucket, prefix, filename):
##     endpoint = os.getenv("AWS_S3_ENDPOINT", "minio.lab.sspcloud.fr")
##     s3_url = f"https://{endpoint}"
##     storage_options = {"client_kwargs": {"endpoint_url": s3_url}}
##     path = f"s3://{bucket}/{prefix}/{filename}"
##     return pd.read_parquet(path, storage_options=storage_options)
## 
## 
## # === NETTOYAGE DE BASE ===
## def clean_games(df):
##     if "game_id" in df.columns:
##         df = df.drop_duplicates(subset=["user_id", "game_id"])
##     else:
##         df = df.drop_duplicates(subset=["user_id", "timestamp"])
## 
##     if "timestamp" not in df.columns and "timestamp_ms" in df.columns:
##         df = df.copy()
##         df.loc[:, "timestamp"] = df["timestamp_ms"]
## 
##     df = df.sort_values(["user_id", "timestamp"])
##     df = df[df["result"] != "Draw"].copy()
## 
##     df["win"] = (df["result"] == "Win").astype(int)
##     df["win_next"] = df.groupby("user_id")["win"].shift(-1)
## 
##     return df
## 
## 
## 
## # === TRANSITIONS & TESTS ===
## def transition_table(df, col="win"):
##     table = pd.crosstab(df[col], df[f"{col}_next"])
##     probs = table.div(table.sum(axis=1), axis=0)
##     return table, probs
## 
## 
## def chi2_pvalue(df, col_x, col_y):
##     table = pd.crosstab(df[col_x], df[col_y])
##     if table.shape == (2, 2):
##         return chi2_contingency(table)[1]
##     return np.nan
## FIN ANCIEN BLOC

from Nettoyage_donnees import clean_games

## Paramètres AWS privés lus dans .env / environnement par config_projet.py.
def load_games_from_s3(bucket, prefix, filename):
    endpoint = os.getenv("AWS_S3_ENDPOINT", "minio.lab.sspcloud.fr")
    if not endpoint.startswith(("http://", "https://")):
        endpoint = "https://" + endpoint
    return pd.read_parquet(f"s3://{bucket}/{prefix}/{filename}", storage_options={"client_kwargs": {"endpoint_url": endpoint}})


def transition_table(df, col="win"):
    table = pd.crosstab(df[col], df[f"{col}_next"]).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    return table, table.div(table.sum(axis=1).replace(0, np.nan), axis=0)


def chi2_pvalue(df, col_x, col_y):
    table = pd.crosstab(df[col_x], df[col_y])
    if table.shape == (2, 2) and (table.sum(axis=0) > 0).all() and (table.sum(axis=1) > 0).all():
        return chi2_contingency(table)[1]
    return np.nan


### I.2 Chargement et préparation des données

In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## df_all = load_games_from_s3(
##     bucket="tristant",
##     prefix="diffusion",
##     filename="dfs_games.parquet"
## )
## 
## n_before = len(df_all)
## df_all = clean_games(df_all)
## n_after = len(df_all)
## 
## print(f"Nombre de parties supprimées (nuls et doublons) : {n_before - n_after}")
## print(f"Nombre de parties restantes : {n_after}")
## 
## df_valid = df_all.dropna(subset=["win_next"])
## FIN ANCIEN BLOC

## PARTIE I — Choisir une source, sans lancer de collecte depuis le notebook.
DEMO = True  ## True = exemple simulé signalé ; False = tes vraies données.
LOCAL_FILE = None  ## Exemple : "Collectes/TON_RUN/Data/dfs_games.parquet"
S3_RUN_ID = ""  ## Identifiant affiché par Master_file.py --upload.

if DEMO:
    ## Exemple reproductible pour vérifier le code ; aucun résultat réel ne doit en être déduit.
    rng = np.random.default_rng(73)
    rows = []
    for u in range(40):
        opening = "Italian Game"
        for t in range(20):
            if rng.random() < .45:
                opening = rng.choice(["Italian Game", "Sicilian Defense", "French Defense"])
            rows.append({"user_id": f"demo_{u}", "game_id": f"demo_{u}_{t}",
                         "timestamp_ms": 1700000000000+t*180000,
                         "result": rng.choice(["Win", "Draw", "Loss"], p=[.46,.08,.46]),
                         "opening_name": opening, "rating": int(rng.integers(1200,1800)),
                         "opponent_rating": int(rng.integers(1200,1800)),
                         "format_partie": rng.choice(["bullet","blitz","rapid","classical"]),
                         "color": rng.choice(["white","black"])})
    df_source = pd.DataFrame(rows)
    print("DONNÉES SIMULÉES — aucune conclusion empirique sur Lichess")
elif LOCAL_FILE:
    df_source = pd.read_parquet(LOCAL_FILE)
elif S3_RUN_ID:
    if Path(S3_RUN_ID).name != S3_RUN_ID or S3_RUN_ID in {".", ".."}:
        raise ValueError("S3_RUN_ID doit être le nom d’une collecte")
    df_source = load_games_from_s3(cfg.S3_BUCKET, cfg.S3_PREFIX + "/" + S3_RUN_ID + "/Data", "dfs_games.parquet")
else:
    raise ValueError("Choisir LOCAL_FILE ou S3_RUN_ID, ou utiliser DEMO=True")

df_all = clean_games(df_source)
df_valid = df_all.dropna(subset=["win_next"]).copy()
print("Lignes joueur-partie :", len(df_all), "| Joueurs :", df_all.user_id.nunique())
print("Nulles conservées :", int(df_all.result.eq("Draw").sum()))
print("Dans les sections historiques II–V, win=0 signifie désormais NON-VICTOIRE (nulle ou défaite).")


Avant d’analyser l’effet Hot Hand, nous effectuons quelques étapes de préparation :

- Tri et déduplication : les parties sont triées par joueur et chronologiquement, et les doublons sont supprimés.

- Codage des résultats : victoire = 1, défaite = 0. Les parties nulles (Draws) sont supprimées pour simplifier le calcul du résultat suivant. Cette simplification n'est bien sûre pas optimale en termes d'analyse mais concerne moins de 5% des parties et simplifie grandement les analyses qui vont suivre.

>⚠️ Exemple : une séquence Win → Draw → Win est comptée comme Win → Win. Les streaks réels sont donc légèrement modifiés.

- Création de win_next : pour chaque joueur, cette colonne correspond au résultat de la partie suivante. Les dernières parties sans win_next sont retirées.

À ce stade, nous ne regardons pas encore les streaks exacts, mais seulement la dépendance entre résultats consécutifs.

### I.3 Prise en main des données

Avant de procéder aux analyses statistiques, nous présentons brièvement
les principales caractéristiques du jeu de données utilisé.

In [ ]:
# Nombre total de parties, joueurs et transitions valides
n_games = len(df_all)
n_players = df_all["user_id"].nunique()
n_transitions = len(df_valid)

print(f"Nombre total de parties : {n_games}")
print(f"Nombre de joueurs uniques : {n_players}")
print(f"Nombre de transitions valides (win_next défini) : {n_transitions}")

# Répartition par format de partie
print("\nRépartition des parties par format :")
print(df_all["format_partie"].value_counts())

# Statistiques de rating
print("\nStatistiques des ratings des joueurs :")
print(df_all["rating"].describe())


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

Le jeu de données contient un nombre suffisant de données pour l'analyse cependant il y a un nombre inférieur de parties bullet et surtout classical, on pourra garder cela en tête pour le reste de l'analyse.

</details>


In [ ]:
games_per_player = df_all.groupby("user_id").size()

games_per_player.describe(percentiles=[0.1, 0.5, 0.9, 0.99])


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

Le jeu de données comprend 35 062 parties jouées par 140 joueurs distincts,
soit en moyenne environ 233 parties par joueur (médiane : 239).

La distribution du nombre de parties par joueur est relativement homogène,
sans asymétrie marquée ni individus dominant excessivement l’échantillon
(nombre maximum de parties par joueur : 237).

Cette structure équilibrée constitue un cadre favorable à l’analyse des
dépendances entre résultats consécutifs, en limitant le risque que les
résultats soient entraînés par un petit nombre de joueurs très actifs.

</details>


## II. Dépendances brutes

### II.1 Dépendance brute entre résultats consécutifs

Nous comparons la probabilité de gagner la partie N+1 selon que la partie N ait été gagnée ou perdue.


In [ ]:
table, probs = transition_table(df_valid)
p_value = chi2_pvalue(df_valid, "win", "win_next")

probs, p_value


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

On observe une probabilité de victoire à N+1 plus élevée après une victoire qu’après une défaite.

Le test du χ² rejette très fortement l’hypothèse d’indépendance entre les résultats consécutifs (p-value < 0,01).

Cependant, cette dépendance brute peut toutefois s’expliquer par des facteurs structurels
(niveau relatif des joueurs, rematchs, sessions de jeu) et ne constitue pas encore une preuve d’un effet *Hot Hand*.


</details>


### II.2 Dépendance brute selon le format de partie

Nous analysons maintenant si la probabilité de gagner la partie suivante dépend du résultat précédent, **en distinguant les formats de partie** (bullet, blitz, rapid, classical).

> On compare P(Victoire N+1 | Victoire N) et P(Victoire N+1 | Défaite N) pour chaque type de format.


#### II.2.A Probabilité de victoire à N+1 selon le résultat précédent et calcul de la p-value

In [ ]:
# Probabilités conditionnelles par format
probs_by_format = (
    df_valid
    .groupby(["format_partie", "win"])["win_next"]
    .mean()
    .unstack()
)

print("Probabilités de victoire à N+1 selon le résultat précédent :")
print(probs_by_format)

# Test χ² par format
print("\nTest d'indépendance χ² par format :")
for fmt, df_fmt in df_valid.groupby("format_partie"):
    p_val = chi2_pvalue(df_fmt, "win", "win_next")
    print(f"Format {fmt} - p-value χ² : {p_val:.3e}")


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

En distinguant les formats, la dépendance entre le résultat en N et en N+1 reste significative pour blitz (p≈0,0069), rapid (p≈8.5xe-6) et classical (p≈4.0xe-4) : la probabilité de gagner en N+1 est systématiquement plus élevée après une victoire qu’après une défaite.

Le bullet fait exception : l’écart va dans le même sens (0,4966 vs 0,4773) mais n’est pas significatif au seuil de 5 % (p≈0,064), possiblement faute d’observations suffisantes et/ou du fait d’une plus grande variabilité dans ce format.

</details>


#### II.2.B Visualisation

In [ ]:
format_order = ["bullet", "blitz", "rapid", "classical"]

# Préparer le DataFrame pour le graphique
prob_df = []
for fmt in format_order:
    if fmt in probs_by_format.index:
        prob_df.append({
            "format": fmt,
            "P(win_next|win)": probs_by_format.loc[fmt, 1],
            "P(win_next|lose)": probs_by_format.loc[fmt, 0]
        })

prob_df = pd.DataFrame(prob_df)

# Graphique simple
prob_df.set_index("format")[["P(win_next|win)", "P(win_next|lose)"]].plot(
    kind="bar", figsize=(8,5), color=["#1f77b4", "#ff7f0e"]
)
plt.ylabel("Probabilité de victoire N+1")
plt.title("Probabilité de victoire selon le format et le résultat précédent")
plt.xticks(rotation=0)
plt.legend(title="Résultat N")
plt.show()

### II.3 Dependance brute selon le niveau du joueur

Nous analysons maintenant si la probabilité de gagner la partie suivante dépend du résultat précédent, **en distinguant le niveau du joueur de partie** que nous classerons dans différents *rating bin*.

> On compare P(Victoire N+1 | Victoire N) et P(Victoire N+1 | Défaite N) pour chaque *rating bin*.



In [ ]:
df_all['rating_bin'] = pd.qcut(df_all['rating'], q=8, duplicates='drop')

df_all['rating_bin'].value_counts().sort_index()


#### II.3.A Probabilité de victoire à N+1 selon le résultat précédent

On calcule la probabilité qu'un joueur gagne la partie suivante en fonction de s'il a gagné ou perdu la précédente, pour chaque tranche de niveau.


In [ ]:
# Reconstruction df_valid
df_valid = df_all.dropna(subset=["win_next"])

# Probabilités conditionnelles par tranche de rating
df_plot = (
    df_valid.groupby(["rating_bin", "win"], observed=True)["win_next"]
            .mean()
            .reset_index()
)
df_plot["win_label"] = df_plot["win"].map({0: "Défaite précédente", 1: "Victoire précédente"})

# Calcul des p-values χ² par tranche de rating
p_values_rating = {}
for bin_label, df_bin in df_valid.groupby("rating_bin", observed = True):
    table = pd.crosstab(df_bin["win"], df_bin["win_next"])
    if table.shape == (2, 2):
        _, p_val, _, _ = chi2_contingency(table)
        p_values_rating[bin_label] = p_val
    else:
        p_values_rating[bin_label] = np.nan

# Affichage rapide
print("Probabilités de victoire à N+1 par tranche de rating :")
print(df_plot.pivot(index="rating_bin", columns="win_label", values="win_next"))

print("\nP-values χ² par tranche de rating :")
for bin_label, p_val in p_values_rating.items():
    print(f"{bin_label}: {p_val:.3e}")


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

Globalement, dans toutes les tranches de rating, la probabilité de gagner en *N+1* est plus élevée après une victoire qu’après une défaite, mais l’effet est hétérogène selon le niveau.

- **Bas niveaux** (677–1146] et (1146–1261] : écart net (≈ +4 à +5 points) et statistiquement significatif (p ≈ 0,001–0,002).
- **Niveaux intermédiaires** : (1261–1358] et surtout (1450–1525] présentent un écart faible et non significatif (p ≈ 0,33 et 0,86).
- **Niveaux élevés** : l’effet réapparaît dans la tranche (1525–1596] avec un écart marqué et significatif (p ≈ 0,003), puis reste positif mais non significatif dans les tranches supérieures (p ≈ 0,056 et 0,13), possiblement en raison d’effectifs plus réduits et d’une performance plus stable.

Dans l’ensemble, la dépendance entre résultats consécutifs semble réelle mais non monotone avec le rating.


</details>


#### II.3.B Visualisation

On trace un graphique en barres montrant la probabilité de victoire à N+1 selon le résultat précédent pour chaque tranche de niveau.


In [ ]:
# --- Probabilités conditionnelles par tranche de rating ---
probs_by_rating = (
    df_all
    .groupby(["rating_bin", "win"], observed=True)["win_next"]
    .mean()
    .unstack()
)

# Préparer le DataFrame pour le graphique
rating_labels = [str(interval) for interval in probs_by_rating.index]
prob_df = []
for i, r in enumerate(probs_by_rating.index):
    prob_df.append({
        "rating_bin": rating_labels[i],
        "P(win_next|win)": probs_by_rating.loc[r, 1],
        "P(win_next|lose)": probs_by_rating.loc[r, 0]
    })

prob_df = pd.DataFrame(prob_df)

# Graphique
prob_df.set_index("rating_bin")[["P(win_next|win)", "P(win_next|lose)"]].plot(
    kind="bar", figsize=(10,5), color=["#1f77b4", "#ff7f0e"]
)
plt.ylabel("Probabilité de victoire N+1")
plt.xlabel("Tranche de niveau (Rating)")
plt.title("Probabilité de victoire selon le niveau du joueur et le résultat précédent")
plt.xticks(rotation=45)
plt.legend(title="Résultat N")
plt.grid(axis='y', alpha=0.3)
plt.show()


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

Même au sein des différentes tranches de niveau, on observe que la probabilité de gagner la partie suivante est plus élevée après une victoire précédente.  

</details>


## III. Proximité temporelle et sessions de jeu

L’effet *Hot Hand* est, par définition, un phénomène **local dans le temps** :  
il suppose que les performances récentes influencent les suivantes **à court terme**.

Dans les analyses précédentes, deux parties consécutives peuvent toutefois être séparées
de quelques secondes… ou de plusieurs jours. Traiter ces situations comme équivalentes
peut créer une dépendance artificielle ou, au contraire, diluer un effet réel.

L’objectif de cette section est donc de **restreindre l’analyse aux parties réellement
consécutives**, jouées dans une même session de jeu.


### III.1 Définition d’une session de jeu et autres variables

Nous définissons une *session de jeu* comme une suite de parties jouées par un même joueur,
séparées par **au plus 30 minutes** entre deux parties consécutives.

Ce seuil constitue un compromis :
- suffisamment court pour capturer des enchaînements intentionnels de parties ;
- suffisamment long pour inclure les pauses naturelles (analyse rapide, attente d’adversaire).

L’analyse repose avant tout sur une **comparaison avant / après filtrage temporel** :
le choix exact du seuil n’est donc pas déterminant.

Nous allons aussi définir de nouvelles variables et dataset filtrés qui nous seront utiles par la suite avant d'alléger la lecture. 

>**L'exécution de cette cellule est donc indispensable au fonctionnement de la suite du notebook :**

In [ ]:
SESSION_GAP = 30  # minutes

# 1) Date + tri
df_all["date"] = pd.to_datetime(df_all["timestamp"], unit="ms")
df_all = df_all.sort_values(["user_id", "date"]).reset_index(drop=True)

# 2) Sessions
df_all["delta_minutes"] = (
    df_all.groupby("user_id")["date"].diff().dt.total_seconds() / 60
)
df_all["new_session"] = (df_all["delta_minutes"] > SESSION_GAP) | (df_all["delta_minutes"].isna())
df_all["session_id"] = df_all.groupby("user_id")["new_session"].cumsum()

# 3) Numéro de partie dans la session (pour fatigue)
df_all["game_in_session"] = df_all.groupby(["user_id", "session_id"]).cumcount() + 1

# 4) Transitions brutes et intra-session
df_all["win_next_brut"] = df_all.groupby("user_id")["win"].shift(-1)

df_all["delta_next_minutes"] = (
    (df_all.groupby("user_id")["date"].shift(-1) - df_all["date"])
    .dt.total_seconds() / 60
)
df_all["win_next_session"] = df_all.groupby("user_id")["win"].shift(-1)
df_all.loc[df_all["delta_next_minutes"] > SESSION_GAP, "win_next_session"] = pd.NA

# 5) Lags intra-session (pour Markov)
df_all["win_lag1"] = df_all.groupby(["user_id", "session_id"])["win"].shift(1)
df_all["win_lag2"] = df_all.groupby(["user_id", "session_id"])["win"].shift(2)

# 6) Couleur
df_all["is_white"] = (df_all["color"] == "white").astype(int)

# 7) Jeux de travail
df_brut = df_all.dropna(subset=["win_next_brut"]).copy()
df_session = df_all.dropna(subset=["win_next_session"]).copy()

df_intra = df_all.dropna(subset=["session_id"]).copy()
df_mk_sess = df_intra.dropna(subset=["win", "win_lag1", "win_lag2"]).copy()
df_mk_sess[["win_lag1", "win_lag2"]] = df_mk_sess[["win_lag1", "win_lag2"]].astype(int)

print("df_brut:", len(df_brut), "| df_session:", len(df_session), "| df_mk_sess:", len(df_mk_sess))

### III.2 Comparaison : dépendance brute vs dépendance intra-session

Nous comparons maintenant la dépendance entre les résultats consécutifs :
- **sans contrainte temporelle** ;
- **en restreignant aux parties intra-session**.


In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## # χ² brute
## table_brut = pd.crosstab(df_brut["win"], df_brut["win_next_brut"])
## _, p_value_brut, _, _ = chi2_contingency(table_brut)
## 
## # χ² intra-session
## table_session = pd.crosstab(df_session["win"], df_session["win_next_session"])
## _, p_value_session, _, _ = chi2_contingency(table_session)
## 
## summary_time = pd.DataFrame({
##     "Cadre d'analyse": ["Sans contrainte temporelle", f"Intra-session (≤ {SESSION_GAP} min)"],
##     "p-value χ²": [p_value_brut, p_value_session]
## })
## summary_time
## FIN ANCIEN BLOC

## Tests exploratoires ; les observations d’un même joueur sont dépendantes.
p_value_brut = chi2_pvalue(df_brut, "win", "win_next_brut")
p_value_session = chi2_pvalue(df_session, "win", "win_next_session")
summary_time = pd.DataFrame({"Cadre": ["Brut", "Intra-session observée"], "p_chi2_exploratoire": [p_value_brut, p_value_session]})
display(summary_time)


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

### III.3 Lecture des résultats

La dépendance entre le résultat d’une partie et celui de la partie suivante
reste hautement significative lorsque l’on restreint l’analyse aux parties
jouées dans une même session (≤ 30 minutes).

Ce résultat indique que la dépendance observée précédemment ne provient pas
uniquement de parties consécutives éloignées dans le temps, mais qu’elle
s’exprime également au sein de séquences de jeu continues.

Le filtrage temporel ne vise pas à faire disparaître l’effet, mais à vérifier
qu’il persiste dans un cadre compatible avec l’hypothèse d’un effet *Hot Hand*
local dans le temps.



</details>


## IV. Mémoire intra-session : analyse markovienne

Dans la section précédente, nous avons montré que la dépendance entre résultats consécutifs
persiste lorsque l’on se restreint à des parties jouées **dans une même session** (≤ 30 minutes).

Nous testons maintenant une question plus précise :

> La dépendance intra-session est-elle **entièrement capturée par le dernier résultat** (Markov d’ordre 1),
> ou existe-t-il une **mémoire plus longue** (Markov d’ordre 2), c.-à-d. un effet cumulatif ?

Formellement : On compare deux hypothèses de mémoire :

- **Ordre 1** : le résultat à *t* dépend uniquement du match précédent (*t−1*).
- **Ordre 2** : le résultat à *t* dépend des deux matchs précédents (*t−1* et *t−2*)

Dans toute cette section, l’analyse est effectuée **uniquement sur les transitions intra-session**.



### IV.1 Probabilités conditionnelles (Markov ordre 2)

On regarde si le résultat d’une partie dépend des **deux résultats précédents**.

Pour chaque partie *t*, on code l’“historique” des deux matchs précédents (*t−2*, *t−1*) parmi 4 états possibles :
- `00` : défaite puis défaite
- `01` : défaite puis victoire
- `10` : victoire puis défaite
- `11` : victoire puis victoire

Pour chacun de ces états, on calcule ensuite la proportion de victoires à *t* :
**P(victoire à t | état passé)**.

> ⚠️ Cette étape est **descriptive** : elle sert à visualiser une éventuelle accumulation.
> La significativité statistique sera testée juste après en comparant l’ordre 1 et l’ordre 2.


In [ ]:
state_order = ["00", "01", "10", "11"]

state2 = df_mk_sess.assign(
    state2=df_mk_sess["win_lag2"].astype(str) + df_mk_sess["win_lag1"].astype(str)
)

markov_table = (
    state2.groupby("state2")["win"]
          .agg(P_win="mean", n="size")
          .reindex(state_order)
)

markov_table


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(markov_table.index, markov_table["P_win"])
plt.ylim(0, 1)
plt.xlabel("État (W_{t-2} W_{t-1})")
plt.ylabel("P(W_t = 1)")
plt.title("Markov ordre 2 (intra-session) : probabilité de victoire selon les 2 résultats précédents")
plt.grid(axis="y", alpha=0.3)
plt.show()


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

La probabilité de victoire augmente avec un historique plus favorable : minimale après `00`, intermédiaire après `01`/`10`, maximale après `11`. Cela est conforme à nos attentes cependant ces résultats restent descriptifs. La section suivante teste formellement si l’ordre 2 apporte un gain par rapport à l’ordre 1.


</details>


### IV.2 Test formel : Markov d’ordre 1 vs Markov d’ordre 2 (LR test)

Nous comparons deux modèles logistiques emboîtés :

- **M1 (ordre 1)** : la probabilité de gagner dépend uniquement du résultat précédent (`win_lag1`).
- **M2 (ordre 2)** : la probabilité de gagner dépend du résultat précédent (`win_lag1`) *et* de l’avant-dernier (`win_lag2`).

Si le modèle M2 améliore significativement l’ajustement, cela signifie que le résultat à *t−2*
apporte une information au-delà du seul résultat à *t−1* : il existe donc une mémoire
plus longue qu’un match.

Nous reportons également les critères **AIC** et **BIC**, qui pénalisent la complexité du modèle.


In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## m1 = smf.logit("win ~ win_lag1", data=df_mk_sess).fit(disp=0)
## m2 = smf.logit("win ~ win_lag1 + win_lag2", data=df_mk_sess).fit(disp=0)
## 
## LR = 2 * (m2.llf - m1.llf)
## df_diff = m2.df_model - m1.df_model
## p_lr = chi2.sf(LR, df_diff)
## 
## summary_models = pd.DataFrame({
##     "Model": ["Order 1", "Order 2"],
##     "LogLik": [m1.llf, m2.llf],
##     "AIC": [m1.aic, m2.aic],
##     "BIC": [m1.bic, m2.bic],
## })
## 
## summary_models, {"LR": LR, "df": df_diff, "p_value": p_lr}
## FIN ANCIEN BLOC

## Comparer deux modèles ADDITIFS à retards ; ce n’est pas l’ordre 2 complètement libre.
try:
    m1 = smf.logit("win ~ win_lag1", data=df_mk_sess).fit(disp=0)
    m2 = smf.logit("win ~ win_lag1 + win_lag2", data=df_mk_sess).fit(disp=0)
    LR = 2 * (m2.llf - m1.llf)
    df_diff = m2.df_model - m1.df_model
    display(pd.DataFrame({"Modèle": ["1 retard", "2 retards additifs"], "AIC": [m1.aic, m2.aic], "BIC": [m1.bic, m2.bic]}))
    print("LR nominal exploratoire :", LR, "p =", chi2.sf(LR, df_diff))
except (ValueError, np.linalg.LinAlgError) as error:
    print("Comparaison non estimable :", error)


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

### IV.3 Interprétation

Le test du rapport de vraisemblance indique que le modèle d’ordre 2 améliore
significativement l’ajustement par rapport au modèle d’ordre 1
(LR = 4,25 ; p-value ≈ 0,039).
Cela suggère que, même conditionnellement au dernier résultat,
l’avant-dernier match contient encore de l’information pour prédire l’issue
de la partie courante.

Toutefois, l’amélioration reste modeste : si le critère AIC favorise
l’ordre 2, le BIC privilégie l’ordre 1.
La mémoire intra-session apparaît donc légèrement supérieure à un match,
mais sans constituer une dynamique fortement persistante.


</details>


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

### IV. Conclusion de section

Au sein des sessions de jeu (parties rapprochées dans le temps), les résultats présentent
une **mémoire d’ordre 2** : les deux matchs précédents contribuent à expliquer la probabilité
de victoire, au-delà du seul dernier résultat.

Cette évidence statistique caractérise une dépendance temporelle plus longue qu’un match,
compatible avec l’idée d’un effet *Hot Hand* cumulatif à court terme.

La suite du notebook introduit des contrôles (niveau du joueur, couleur, fatigue/session)
afin d’évaluer dans quelle mesure cette mémoire peut être expliquée par des facteurs alternatifs.


</details>


## V. Contrôles et mécanismes

Après avoir mis en évidence une dépendance intra-session entre résultats consécutifs,
ainsi qu’une mémoire d’ordre 2, cette section poursuit deux objectifs :

1. **Tester la robustesse** de cette dépendance en contrôlant pour des facteurs observables.
2. **Explorer des mécanismes plausibles** susceptibles de générer cette dynamique,
   sans prétendre à une identification causale.

Cette section ne vise pas à “prouver” l’effet *Hot Hand*,
mais à vérifier qu’il ne se réduit pas à des artefacts évidents
et à proposer des interprétations compatibles avec les résultats observés.


### V.1 Contrôle statistique : régression logistique (niveau et couleur)

Même au sein des sessions, la dépendance observée pourrait s’expliquer par :
- le niveau intrinsèque du joueur ;
- un avantage structurel (jouer avec les Blancs).

Nous estimons donc un modèle logistique où la probabilité de victoire dépend :
- du résultat précédent ;
- du rating du joueur (proxy du niveau) ;
- de la couleur.

L’objectif est de vérifier si l’effet du résultat précédent persiste
*toutes choses égales par ailleurs*.


In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## df_reg = df_mk_sess.dropna(subset=["rating", "is_white"]).copy()
## 
## model = smf.logit("win ~ win_lag1 + rating + is_white", data=df_reg).fit(disp=0)
## model.summary()
## FIN ANCIEN BLOC

## Conserver la régression initiale comme comparaison historique, sans conclusion causale.
df_reg = df_all.dropna(subset=["rating", "is_white", "win_lag1"]).copy()
try:
    model = smf.logit("win ~ win_lag1 + rating + is_white", data=df_reg).fit(disp=0)
    display(model.summary())
    print("p-values nominales historiques ; voir la régression groupée par joueur en VI.")
except (ValueError, np.linalg.LinAlgError) as error:
    print("Modèle historique non estimable :", error)


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

La régression logistique est estimée sur 12 033 observations intra-session.

Après contrôle du niveau du joueur (rating) et de la couleur, le résultat précédent reste
positivement associé à la probabilité de victoire : le coefficient de `win_lag1` est positif
et  significatif (coef ≈ 0,123 ; p < 0,001).

L’effet “Blancs” est également positif et significatif (coef ≈ 0,125 ; p < 0,001), ce qui est
cohérent avec un avantage structurel de la couleur.

En revanche, le rating n’apparaît pas significatif (coeff ≈ 0 ; p ≈ 0,53) dans ce modèle,
ce qui suggère que, dans cet échantillon intra-session, le niveau n’explique pas
la dépendance temporelle observée (ou qu’il est déjà partiellement neutralisé par le matchmaking).

Ainsi, l’association entre victoire récente et victoire suivante persiste même en tenant compte
de facteurs observables simples, ce qui renforce la robustesse du résultat.

</details>


### V.2 Dynamique de session : fatigue et relâchement

Une autre explication possible de la dépendance observée est une dynamique de session :
au fil des parties, la performance peut évoluer en raison de la fatigue,
du relâchement ou du tilt.

Nous examinons ici comment le taux de victoire évolue
en fonction de la position de la partie dans la session.


In [ ]:
df_fatigue = df_intra.loc[df_intra["game_in_session"].le(15)]

rate_by_pos = df_fatigue.groupby("game_in_session")["win"].mean()

plt.figure(figsize=(7,4))
plt.plot(rate_by_pos.index, rate_by_pos.values, marker="o")
plt.axhline(0.5, linestyle="--", alpha=0.5)
plt.xlabel("Numéro de la partie dans la session")
plt.ylabel("Taux de victoire")
plt.title("Performance au fil de la session")
plt.grid(axis="y", alpha=0.3)
plt.show()

> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

On observe une légère inflexion de la performance au cours des sessions, avec un creux autour de la 10ᵉ partie jouée, ce qui est compatible avec un effet de fatigue, de relâchement ou de tilt.

Le taux de victoire remonte ensuite assez nettement sur les dernières positions. Il serait tentant d’y voir un “rebond” après avoir tenu bon, mais cette lecture est très probablement trompeuse : plus on avance dans la session, plus on ne conserve que les sessions longues (et donc un sous-ensemble particulier de joueurs / moments), et surtout le nombre d’observations chute fortement.

Nous pouvons le vérifier directement en affichant, pour chaque rang dans la session, le nombre de parties utilisées pour calculer la moyenne :

</details>


In [ ]:
fatigue_stats = (
    df_fatigue
    .groupby("game_in_session")
    .agg(
        win_rate=("win", "mean"),
        n_games=("win", "size")
    )
)

fatigue_stats

> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

Comme attendu, nous avons trop peu de parties pour les séries longues pour en tirer des conclusions pertinentes.

</details>


### V.3 Mécanisme comportemental : rejouer la même ouverture

Un mécanisme plausible de persistance des performances est l’adaptation des décisions.
Après une victoire, les joueurs peuvent être enclins à répéter une stratégie perçue
comme efficace.

Nous testons ici si la probabilité de rejouer la même ouverture
dépend du résultat de la partie précédente.


In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## df_open = df_all.assign(
##     same_opening_next=lambda d: d.groupby("user_id")["opening_name"].shift(-1).eq(d["opening_name"])
## ).dropna(subset=["win", "same_opening_next"])
## 
## prob_opening = df_open.groupby("win")["same_opening_next"].mean()
## 
## tab_open = pd.crosstab(df_open["win"], df_open["same_opening_next"])
## p_open = chi2_contingency(tab_open)[1]
## 
## prob_opening, tab_open, p_open
## FIN ANCIEN BLOC

from Analyse_repetition_ouverture import creer_transitions

## Un nouveau DataFrame de transitions, où chaque ligne possède une vraie partie suivante.
df_open = creer_transitions(df_all)
df_open = df_open[df_open["known_pair"]].copy()
df_open["same_opening_next"] = df_open["repeat_opening"]
prob_opening = df_open.groupby("win")["same_opening_next"].mean()
tab_open = pd.crosstab(df_open["win"], df_open["same_opening_next"])
p_open = chi2_pvalue(df_open, "win", "same_opening_next")
display(prob_opening, tab_open)
print("Ancienne question : après victoire, répète-t-on ? p exploratoire =", p_open)


In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## # Test d'indépendance
## table_open = pd.crosstab(df_open["win"], df_open["same_opening_next"])
## _, p_open, _, _ = chi2_contingency(table_open)
## 
## table_open, p_open
## FIN ANCIEN BLOC

## Le test vient d’être calculé : éviter de l’exécuter une seconde fois.
## La nouvelle question « répéter puis gagner » est étudiée en VI.
display(tab_open)


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

On observe que les joueurs rejouent légèrement plus souvent la même ouverture après une victoire
qu’après une défaite : 14,66 % contre 13,36 %.

Le test du χ² rejette l’hypothèse d’indépendance entre le résultat de la partie et le fait de
rejouer la même ouverture à la partie suivante (p ≈ 5,5e-4).

L’écart reste toutefois modéré en valeur absolue : cela suggère un mécanisme comportemental
plausible (répéter une stratégie “qui vient de marcher”), sans pour autant suffire à expliquer
à lui seul la dépendance globale entre résultats.


</details>


### V.4 Null model et simulation

Afin d’évaluer si la dépendance observée entre résultats consécutifs intra-session peut s’expliquer par le hasard, nous comparons la statistique observée à sa distribution sous un null model.  

La statistique étudiée est **Δ = P(victoire à t+1 | victoire à t) − P(victoire à t+1 | défaite à t)**, c’est-à-dire l’écart de probabilité de gagner la partie suivante selon que la partie courante a été gagnée ou perdue.
  
Ce null model conserve, pour chaque joueur et chaque session, le nombre de victoires et la longueur des sessions, mais randomise l’ordre des résultats à l’intérieur de celles-ci.


In [ ]:
## DEBUT ANCIEN BLOC — conservé, non exécuté
## SEED = 0
## N_SIM = 500 
## rng = np.random.default_rng(SEED)
## 
## df_sim = (
##     df_intra[["user_id", "session_id", "date", "win"]]
##     .dropna()
##     .sort_values(["user_id", "session_id", "date"])
##     .reset_index(drop=True)
## )
## 
## def compute_delta(df, col="win"):
##     win_next = df.groupby(["user_id", "session_id"], sort=False)[col].shift(-1)
##     tmp = df.assign(win_next=win_next).dropna(subset=["win_next"])
##     return (
##         tmp.loc[tmp[col] == 1, "win_next"].mean()
##         - tmp.loc[tmp[col] == 0, "win_next"].mean()
##     )
## 
## real_delta = compute_delta(df_sim, col="win")
## 
## idx_by_group = [
##     g.index.to_numpy()
##     for _, g in df_sim.groupby(["user_id", "session_id"], sort=False)
## ]
## 
## wins = df_sim["win"].to_numpy()
## sim_deltas = np.empty(N_SIM)
## 
## for i in range(N_SIM):
##     shuffled = wins.copy()
##     for idx in idx_by_group:
##         shuffled[idx] = rng.permutation(shuffled[idx])
##     df_sim["win_perm"] = shuffled
##     sim_deltas[i] = compute_delta(df_sim, col="win_perm")
## 
## df_sim.drop(columns=["win_perm"], inplace=True)
## 
## #résultats
## p_val = (sim_deltas >= real_delta).mean()
## real_delta, p_val
## print(f"Δ observé (intra-session) = {real_delta:.5f}")
## print(f"p-value (permutation intra-session) < {1/N_SIM:.3f}" if p_val == 0 else f"p-value = {p_val:.4f}")
## 
## #histogramme
## plt.figure(figsize=(8,4))
## plt.hist(sim_deltas, bins=35, density=True, alpha=0.9, edgecolor="white")
## plt.axvline(real_delta, linestyle="--", linewidth=2, label=f"Observé ({real_delta:.5f})")
## plt.xlabel("Δ intra-session")
## plt.ylabel("Densité")
## plt.title("Null model intra-session — permutation des résultats")
## plt.legend(frameon=False)
## plt.grid(axis="y", alpha=0.25)
## plt.tight_layout()
## plt.show()
## FIN ANCIEN BLOC

SEED = 0
N_SIM = 500 
rng = np.random.default_rng(SEED)

df_sim = (
    df_intra[["user_id", "session_id", "date", "win"]]
    .dropna()
    .sort_values(["user_id", "session_id", "date"])
    .reset_index(drop=True)
)

def compute_delta(df, col="win"):
    win_next = df.groupby(["user_id", "session_id"], sort=False)[col].shift(-1)
    tmp = df.assign(win_next=win_next).dropna(subset=["win_next"])
    return (
        tmp.loc[tmp[col] == 1, "win_next"].mean()
        - tmp.loc[tmp[col] == 0, "win_next"].mean()
    )

real_delta = compute_delta(df_sim, col="win")

idx_by_group = [
    g.index.to_numpy()
    for _, g in df_sim.groupby(["user_id", "session_id"], sort=False)
]

wins = df_sim["win"].to_numpy()
sim_deltas = np.empty(N_SIM)

for i in range(N_SIM):
    shuffled = wins.copy()
    for idx in idx_by_group:
        shuffled[idx] = rng.permutation(shuffled[idx])
    df_sim["win_perm"] = shuffled
    sim_deltas[i] = compute_delta(df_sim, col="win_perm")

df_sim.drop(columns=["win_perm"], inplace=True)

#résultats
p_val = (1 + np.count_nonzero(sim_deltas >= real_delta)) / (N_SIM + 1)
real_delta, p_val
print(f"Δ observé (intra-session) = {real_delta:.5f}")
print(f"p-value (permutation intra-session) < {1/N_SIM:.3f}" if p_val == 0 else f"p-value = {p_val:.4f}")

#histogramme
plt.figure(figsize=(8,4))
plt.hist(sim_deltas, bins=35, density=True, alpha=0.9, edgecolor="white")
plt.axvline(real_delta, linestyle="--", linewidth=2, label=f"Observé ({real_delta:.5f})")
plt.xlabel("Δ intra-session")
plt.ylabel("Densité")
plt.title("Null model intra-session — permutation des résultats")
plt.legend(frameon=False)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

La distribution issue des permutations est fortement concentrée autour de 0,015, tandis que la valeur observée de la statistique se situe nettement dans la queue droite de cette distribution, autour de 0,030.

Aucune permutation ne produit une valeur aussi élevée (p-value < 0,002), ce qui indique l’existence d’une dépendance locale entre résultats consécutifs au sein des sessions.

Le Δ est globalement positif, ce qui reflète un taux de victoire moyen légèrement supérieur à 50 %. Cette asymétrie s’explique par la sélection d’un sous-ensemble de joueurs plutôt que l’ensemble de la plateforme, en particulier des joueurs plus actifs et engagés, caractérisés par des sessions comportant un nombre élevé de parties. Cette caractéristique est conservée dans le null model et n’affecte pas l’interprétation des dépendances conditionnelles étudiées.

</details>


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

### V.5 Conclusion de la section

Les contrôles montrent que l’association entre victoire récente et probabilité de victoire suivante persiste après prise en compte du niveau du joueur et de la couleur, ce qui suggère qu’elle ne se réduit pas à des facteurs structurels simples.

Les analyses complémentaires pointent des mécanismes plausibles (légère tendance à répéter une stratégie après une victoire, dynamique de session compatible avec fatigue/relâchement), cohérents avec une dépendance à court terme au sein des sessions.

Enfin, le test de permutation intra-session, qui conserve pour chaque joueur et chaque session le nombre de victoires et la longueur des sessions tout en randomisant l’ordre des résultats, rejette l’hypothèse d’une absence de dépendance temporelle (p-value < 0,002). Cela confirme que l’effet détecté ne provient pas uniquement de la composition des sessions ou des différences de niveau, mais d’une dynamique locale entre parties consécutives.

Au total, les résultats sont compatibles avec une dynamique temporelle réelle mais subtile, proche d’un Hot Hand limité à court terme, sans implication causale stricte.


</details>


> **Ancienne rédaction conservée pour comparaison.** Ces chiffres et conclusions ne sont pas validés pour la nouvelle collecte. Les résultats de l’extension se lisent en partie VI.

<details><summary>Lire le commentaire historique</summary>

## Conclusion générale

Ce notebook a étudié l’existence d’un effet de *Hot Hand* dans des parties d’échecs en ligne,
défini comme une augmentation de la probabilité de gagner une partie après une victoire récente.

Les résultats montrent d’abord une dépendance nette entre résultats consécutifs : la probabilité
de victoire à N+1 est plus élevée après une victoire qu’après une défaite, et ce constat se retrouve
quel que soit le format de partie et la tranche de niveau.

Un point central de l’analyse est la prise en compte de la proximité temporelle. En restreignant
l’étude aux transitions intra-session (parties séparées de 30 minutes au plus), la dépendance
demeure hautement significative. Cela indique que l’effet observé n’est pas uniquement dû à des
parties “consécutives” mais espacées dans le temps, et qu’il s’exprime également dans des séquences
de jeu continues.

L’analyse markovienne renforce ce constat : au sein des sessions, un modèle qui tient compte des
deux résultats précédents (ordre 2) améliore significativement l’ajustement par rapport à un modèle
ne dépendant que du dernier résultat (ordre 1). Autrement dit, la dynamique intra-session ne se
réduit pas à une mémoire d’un seul match.

Par ailleurs, plusieurs contrôles suggèrent que cette dépendance ne se résume pas à des facteurs structurels
simples. Dans une régression logistique, l’association entre victoire récente et victoire suivante
persiste après contrôle du niveau du joueur et de la couleur. De plus, on observe une légère tendance
à répéter une ouverture après une victoire, compatible avec des ajustements de décision à court terme.

Enfin, le test de permutation intra-session, qui conserve pour chaque joueur et chaque session le nombre de victoires et la longueur des sessions tout en randomisant l’ordre des résultats, rejette l’hypothèse d’une absence de dépendance temporelle. Ce résultat indique que l’effet détecté ne s’explique pas uniquement par la composition des sessions ou l’hétérogénéité des joueurs, mais correspond bien à une dépendance locale entre parties consécutives au sein des sessions.

En résumé, les données sont compatibles avec l’existence d’une dynamique de performance à court terme au sein des sessions, cohérente avec une forme modérée d’effet Hot Hand. Néanmoins, cette dynamique apparaît fragile et sensible au choix des outils statistiques. L’étude restant observationnelle, l’absence d’information détaillée sur l’adversaire et le score attendu limite toute interprétation causale stricte. Une extension naturelle consisterait à intégrer ces dimensions et à tester la robustesse des résultats à d’autres définitions de session et d’autres statistiques de dépendance temporelle.


</details>


## VI. Rôle de la répétition d’ouverture dans les victoires successives

### VI.1 Identifier à 5 ou 10 coups et créer les DataFrames

Le référentiel officiel rassemble a.tsv à e.tsv. On reconstruit la partie jusqu’à **10 ou 20 demi-coups**. La dernière position nommée atteinte est conservée avec sa profondeur réelle ; elle peut correspondre à une famille plus générale. Le nom API reste séparé et ne remplace pas cette analyse limitée en profondeur.

Une **colonne** contient une valeur par ligne. Un **DataFrame** rassemble plusieurs lignes et colonnes. `opening_name_next` donne une seule ouverture suivante sur chaque ligne ; `apres_victoire` sera le DataFrame des transitions dont la partie courante est gagnée.


In [ ]:
from Scripts_generation_data.Creation_Df_openings import creer_openings, identify_games
from Analyse_repetition_ouverture import preparer_cadres, taux_apres_victoire

OPENING_MOVES = 10  ## Choisir 5 ou 10 coups complets.
if OPENING_MOVES not in (5, 10):
    raise ValueError("Choisir 5 ou 10")
cfg.create_run()  ## Chaque exécution garde ses propres fichiers.
if "moves" in df_source and df_source["moves"].notna().any():
    reference = creer_openings()
    analyse_parties = identify_games(df_source, reference, max_fullmoves=OPENING_MOVES, min_fullmoves=5)
    analyse_parties = clean_games(analyse_parties)
else:
    analyse_parties = clean_games(df_source)  ## Ne pas exporter les colonnes de tranches Interval ajoutées pour le rapport historique.
    print("Pas de coups : les noms existants sont utilisés sans valider leur profondeur. DEMO =", DEMO)

cadres = preparer_cadres(analyse_parties)
display(pd.Series({name: len(table) for name, table in cadres.items()}, name="transitions"))
apres_victoire, taux = taux_apres_victoire(cadres["immediat"])
display(apres_victoire[["user_id", "game_id", "game_id_next", "result", "result_next", "opening_name", "opening_name_next", "repeat_opening", "color", "color_next"]].head(10))
display(taux)


### VI.2 Définir le χ², puis le test exact de Fisher

H0 : parmi les transitions après une victoire, répétition et victoire suivante sont indépendantes. H1 : elles sont associées. La statistique de Pearson additionne `(observé − attendu)² / attendu`. Dans un tableau 2×2, les degrés de liberté valent 1. Les attendus doivent être suffisamment grands ; le V de Cramér décrit la taille de l’association.

Fisher exact est proposé sur le tableau 2×2, notamment si les effectifs sont faibles. Il n’est pas une régression multivariable. **Ni Fisher ni le χ² simple ne rendent indépendants les matchs du même joueur.** Leurs p-values groupées restent donc exploratoires.


In [ ]:
from Analyse_repetition_ouverture import tester_table

resultat, observe, attendu = tester_table(apres_victoire)
display(observe.rename_axis(index="Répéter : 0 non / 1 oui", columns="Gagner suivante : 0 non / 1 oui"))
display(attendu)
display(pd.Series(resultat))


### VI.3 Appliquer la même question dans trois cadres

1. **Immédiat** : après victoire, comparer répéter/changer et gagner la suivante observée.
2. **Immédiat même couleur** : même comparaison uniquement lorsque les deux parties immédiatement successives ont la même couleur.
3. **Prochaine même couleur** : comparer à la prochaine partie de cette couleur ; il peut y avoir des parties entre les deux. Cela décrit le répertoire par couleur, pas une suite immédiate de victoires.

Une nulle suivante compte comme non-victoire. Une ouverture inconnue ne compte pas comme changement. Les trois cadres sont annoncés avant de voir les p-values ; Holm corrige cette famille de comparaisons séparément pour χ² et Fisher. Le bootstrap rééchantillonne les **comptes entiers**, avec au moins 20 comptes comme garde-fou pédagogique, sans garantie de validité pour tous les réseaux d’adversaires.


In [ ]:
from Analyse_repetition_ouverture import analyser

resultats = analyser(analyse_parties, cfg.RESULTS_DIR, run_regression=cfg.RUN_REGRESSIONS)
display(resultats["tests"])
display(resultats["bootstrap"])
for name, table in resultats["taux"].items():
    print(name)
    display(table)


### VI.4 Régression multivariable pour la question principale

On utilise les transitions immédiates après victoire. Le modèle logistique explique la victoire suivante par répétition, écart Elo **avant** la suivante, couleur et cadence de la suivante. Il conserve une structure proche de la logistique déjà présente dans ton projet.

Les erreurs-types sont regroupées par joueur. Un coefficient positif de répétition décrit une association ajustée ; `exp(coefficient)` est un odds ratio, pas un gain direct en points de pourcentage. Avec cinq comptes seulement, le modèle est volontairement déclaré non estimable pour l’inférence groupée ; les descriptifs restent disponibles. Si l’Elo adverse manque dans un ancien parquet, il faut compléter la collecte, pas le remplacer par ratingDiff.

Cette première régression teste la répétition. Le classement des familles d’ouvertures nécessiterait une extension et davantage d’effectifs par famille ; les descriptifs par famille et couleur restent disponibles.


In [ ]:
display(pd.Series(resultats["regression"]))
display(resultats["coefficients"])


### VI.5 Statistiques descriptives et graphiques simples

`.mean()` calcule une moyenne ; `.std()` l’écart-type d’échantillon ; `.min()`/`.max()` donnent les extrêmes. Le score vaut 1/0,5/0 pour victoire/nulle/défaite et sa moyenne n’est donc pas exactement un taux de victoire. Chaque taux doit être accompagné de son effectif.

Un histogramme décrit la distribution des classements. Les barres compareront les résultats ou les taux, sans prétendre que l’ouverture cause la victoire. Une famille rare avec deux victoires sur deux parties n’est pas automatiquement meilleure.


In [ ]:
from Analyse_repetition_ouverture import descriptifs

display(descriptifs(analyse_parties))
print("Moyenne du score :", analyse_parties["score"].mean())
print("Écart-type du score :", analyse_parties["score"].std())
print("Maximum du score :", analyse_parties["score"].max())
display(pd.read_csv(cfg.RESULTS_DIR / "descriptifs_ouvertures_couleur.csv"))
for filename in ["histogramme_rating.png", "resultats.png", "taux_immediat.png"]:
    path = cfg.RESULTS_DIR / filename
    if path.exists():
        display(Image(filename=str(path)))
print("Résultats conservés dans", cfg.RESULTS_DIR)


### VI.6 Interpréter et conserver les résultats

Commencer par les effectifs et l’écart de taux répéter − changer, puis lire les incertitudes. Vérifier les ouvertures inconnues et les parties exclues. Les résultats DEMO sont fictifs. Pour conclure sur Lichess, relancer sur les vraies données et rédiger l’interprétation depuis les nouveaux tableaux.

Les fichiers locaux et les anciennes collectes S3 restent conservés. Le notebook n’envoie rien automatiquement ; l’option `--upload` du Master ou l’appel explicite ci-dessous enregistre la nouvelle collecte sur le bucket Hadrien.


In [ ]:
UPLOAD_S3 = False
if UPLOAD_S3:
    ## Sauvegarder la source et les résultats de CETTE exécution avant l’envoi.
    analyse_parties.to_parquet(cfg.DATA_DIR / "dfs_games.parquet", index=False)
    from Stockage_SSPCloud import upload_folder_to_s3
    upload_folder_to_s3(cfg.RUN_DIR)
